# Constellation v2 — Detector training on Colab

From-scratch hand+head detector (Stage 1). Runs the SAME `aslv2.detect.train` code as local; only the device (CUDA) and data root differ.

## Before running — upload these **2 files** to one Drive folder (default `MyDrive/asl-detector/`):
1. `colab_detector_code.zip`  (from `model-v2/artifacts/`, run `python scripts/package_for_colab.py` to make it) — code + manifests (including `*_small.json`)
2. `detect_small.zip`         (from `model-v2/artifacts/`)  — shrunk images, ~1 GB

Total upload ~1 GB (replaces the old ~10.5 GB upload). You do **not** need the original full-res zips.

**Runtime → Change runtime type → GPU (A100/L4/T4).**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR  = '/content/drive/MyDrive/asl-detector'   # <-- the folder you uploaded the 2 files to
DATA_ROOT  = '/content/data/detect_small'
CODE_DIR   = '/content/model-v2'
assert os.path.isdir(DRIVE_DIR), f'Upload the 2 files to {DRIVE_DIR} first'
print('Drive folder contents:', os.listdir(DRIVE_DIR))

In [ ]:
# Copy zips from Drive to fast local Colab disk, then extract.
# detect_small.zip contains 100doh/... and widerface/... directly,
# so extract straight into DATA_ROOT.
import shutil
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(CODE_DIR, exist_ok=True)

for z in ['detect_small.zip', 'colab_detector_code.zip']:
    src = f'{DRIVE_DIR}/{z}'
    print('copying', z, '...')
    shutil.copy(src, f'/content/{z}')
print('copied all zips to /content')

In [ ]:
# Extract: detect_small.zip -> DATA_ROOT (100doh/... and widerface/... land directly there)
!unzip -q -o /content/detect_small.zip         -d $DATA_ROOT
!unzip -q -o /content/colab_detector_code.zip  -d $CODE_DIR

# sanity: a manifest-relative path must resolve under DATA_ROOT
import json
m = json.load(open(f'{CODE_DIR}/artifacts/detect/val_small.json'))
p = os.path.join(DATA_ROOT, m[0]['image'])
print('sample image resolves:', os.path.exists(p), '->', p)

In [ ]:
!pip install -q -e $CODE_DIR
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

In [ ]:
# Train. device() auto-selects CUDA on Colab. Checkpoints land in artifacts/checkpoints/detector/.
!cd $CODE_DIR && python -m aslv2.detect.train --config configs/detector_colab.yaml --data-root $DATA_ROOT

In [ ]:
# Persist the trained detector back to Drive (so it survives the session).
ckpt = f'{CODE_DIR}/artifacts/checkpoints/detector'
for f in ['best.pt', 'history.json']:
    src = f'{ckpt}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_DIR}/{f}')
        print('saved to Drive:', f)

import json
h = json.load(open(f'{ckpt}/history.json'))
print('best val:', h.get('best_score'), '| final epoch metrics:', h['history'][-1] if h.get('history') else None)
print('\nDownload best.pt from Drive into model-v2/artifacts/checkpoints/detector/ on your machine.')